# CPE 494 ERP Systems with AI Enhancements
## Lab 1 Chatbot with RAG.

### 1. Environment Setup

#### Install Vertex AI SDK and Google GEN AI SDK

In [1]:
#%pip install --upgrade --quiet google-cloud-aiplatform google-genai

# Install the necessary libraries
!pip install --upgrade --quiet google-cloud-aiplatform google-genai

# Automatically restart the kernel to pick up the new library versions
import os
os.kill(os.getpid(), 9)

: 

: 

: 

#### Authenticate notebook env (Colab Kernel Only)

In [1]:
import sys

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()

#### Set Google Cloud project information and initialize Vertex AI SDK

In [2]:
import os

import vertexai
from google import genai

# fmt: off
PROJECT_ID = "local-disk-478816-f6"  # @param {type: "string", placeholder: "[your-project-id]", isTemplate: true} #CHANGE ME
# fmt: on
if not PROJECT_ID or PROJECT_ID == "[your-project-id]":
    PROJECT_ID = str(os.environ.get("GOOGLE_CLOUD_PROJECT"))

# See https://cloud.google.com/vertex-ai/generative-ai/docs/rag-engine/rag-overview#supported-regions for location options.
vertexai.init(project=PROJECT_ID, location="asia-southeast1")
client = genai.Client(vertexai=True, project=PROJECT_ID, location="asia-southeast1")

#### Import Libraries

In [3]:
from IPython.display import Markdown, display
from google.genai.types import GenerateContentConfig, Retrieval, Tool, VertexRagStore
from vertexai import rag

### 2. Embedding & Vector Database (Cloud-Native)

#### Create a Rag Corpus

In [5]:
# Currently supports Google first-party embedding models
# fmt: off
EMBEDDING_MODEL_005 = "publishers/google/models/text-embedding-005"  # @param {type:"string", isTemplate: true}
EMBEDDING_MODEL_002 = "publishers/google/models/text-multilingual-embedding-002"  # @param {type:"string", isTemplate: true}
# fmt: on

rag_corpus = rag.create_corpus(
    display_name="my-rag-corpus",
    backend_config=rag.RagVectorDbConfig(
        rag_embedding_model_config=rag.RagEmbeddingModelConfig(
            vertex_prediction_endpoint=rag.VertexPredictionEndpoint(
                publisher_model=EMBEDDING_MODEL_002
            )
        )
    ),
)

#### Check the corpus just created

In [6]:
rag.list_corpora()

ListRagCorporaPager<rag_corpora {
  name: "projects/local-disk-478816-f6/locations/asia-southeast1/ragCorpora/4611686018427387904"
  display_name: "my-rag-corpus"
  create_time {
    seconds: 1771690276
    nanos: 267193000
  }
  update_time {
    seconds: 1771690276
    nanos: 267193000
  }
  corpus_status {
    state: ACTIVE
  }
  vector_db_config {
    rag_managed_db {
      knn {
      }
    }
    rag_embedding_model_config {
      vertex_prediction_endpoint {
        endpoint: "projects/local-disk-478816-f6/locations/asia-southeast1/publishers/google/models/text-multilingual-embedding-002"
      }
    }
  }
}
rag_corpora {
  name: "projects/local-disk-478816-f6/locations/asia-southeast1/ragCorpora/5764607523034234880"
  display_name: "my-rag-corpus"
  create_time {
    seconds: 1771693109
    nanos: 840436000
  }
  update_time {
    seconds: 1771693109
    nanos: 840436000
  }
  corpus_status {
    state: ACTIVE
  }
  vector_db_config {
    rag_managed_db {
      knn {
      }
   

#### Upload a local file to the corpus

In [7]:
%%writefile test.md

Retrieval-Augmented Generation (RAG) is a technique that enhances the capabilities of large language models (LLMs) by allowing them to access and incorporate external data sources when generating responses. Here's a breakdown:

**What it is:**

* **Combining Retrieval and Generation:**
    * RAG combines the strengths of information retrieval systems (like search engines) with the generative power of LLMs.
    * It enables LLMs to go beyond their pre-trained data and access up-to-date and specific information.
* **How it works:**
    * When a user asks a question, the RAG system first retrieves relevant information from external data sources (e.g., databases, documents, web pages).
    * This retrieved information is then provided to the LLM as additional context.
    * The LLM uses this augmented context to generate a more accurate and informative response.

**Why it's helpful:**

* **Access to Up-to-Date Information:**
    * LLMs are trained on static datasets, so their knowledge can become outdated. RAG allows them to access real-time or frequently updated information.
* **Improved Accuracy and Factual Grounding:**
    * RAG reduces the risk of LLM "hallucinations" (generating false or misleading information) by grounding responses in verified external data.
* **Enhanced Contextual Relevance:**
    * By providing relevant context, RAG enables LLMs to generate more precise and tailored responses to specific queries.
* **Increased Trust and Transparency:**
    * RAG can provide source citations, allowing users to verify the information and increasing trust in the LLM's responses.
* **Cost Efficiency:**
    * Rather than constantly retraining large language models, RAG allows for the introduction of new data in a more cost effective way.

In essence, RAG bridges the gap between the vast knowledge of LLMs and the need for accurate, current, and contextually relevant information.


Writing test.md


In [8]:
print("Chunking of file test.md ...")
rag_file = rag.upload_file(
    corpus_name=rag_corpus.name,
    path="test.md",
    display_name="test.md",
    description="my test file",
    #next line on transformation_config is optional, so it's been removed to use the default.
    #transformation_config=rag.TransformationConfig(
    #    chunking_config=rag.ChunkingConfig(chunk_size=512, chunk_overlap=50))
)
print("Chunking of test.md done.")

Chunking of file test.md ...
Chunking of test.md done.


#### Import files from Google Cloud Storage

In [9]:
# 1. Create the folder first so gsutil knows it's a directory
!mkdir -p demo_pdfs

# 2. Copy the files from the public bucket into that folder
# We use quotes and the star (*) to ensure we get the files inside the folder
!gsutil -m cp -r "gs://cpe-kmutt-leb1-rag-966535060840/*" ./demo_pdfs/

# 3. Verify the files are there
!ls ./demo_pdfs

Copying gs://cpe-kmutt-leb1-rag-966535060840/ปร.ด.วิศวกรรมไฟฟ้าและคอมพิวเตอร์ (หลักสูตรนานาชาติ)-PhD International _2565.pdf...
Copying gs://cpe-kmutt-leb1-rag-966535060840/วศ.บ.วิศวกรรมคอมพิวเตอร์-BE Regular TH_2564.pdf...
Copying gs://cpe-kmutt-leb1-rag-966535060840/วศ.บ.วิศวกรรมคอมพิวเตอร์ (หลักสูตรนานาชาติ)-BE International_2564.pdf...
Copying gs://cpe-kmutt-leb1-rag-966535060840/วศ.ม._วท.ม.วิศวกรรมคอมพิวเตอร์ (หลักสูตรนานาชาติ)-MEng_MSc International_2565.pdf...
| [4/4 files][  1.2 MiB/  1.2 MiB] 100% Done                                    
Operation completed over 4 objects/1.2 MiB.                                      
'ปร.ด.วิศวกรรมไฟฟ้าและคอมพิวเตอร์ (หลักสูตรนานาชาติ)-PhD International _2565.pdf'
'วศ.บ.วิศวกรรมคอมพิวเตอร์-BE Regular TH_2564.pdf'
'วศ.บ.วิศวกรรมคอมพิวเตอร์ (หลักสูตรนานาชาติ)-BE International_2564.pdf'
'วศ.ม._วท.ม.วิศวกรรมคอมพิวเตอร์ (หลักสูตรนานาชาติ)-MEng_MSc International_2565.pdf'


In [10]:
#reading from a public bucket.  This bucket is hosted by someone else so it's free.
INPUT_GCS_BUCKET = "gs://cpe-kmutt-leb1-rag-966535060840/"

print("Start reading and chunking data in gcs bucket ", INPUT_GCS_BUCKET, " ...")

response = rag.import_files(
    corpus_name=rag_corpus.name,
    paths=[INPUT_GCS_BUCKET],
    # Optional
    transformation_config=rag.TransformationConfig(
        chunking_config=rag.ChunkingConfig(chunk_size=1024, chunk_overlap=128)
    ),
    # max_embedding_requests_per_min=900,  # Optional
)
print("Chunking of data in Public GCS Bucket done.")

Start reading and chunking data in gcs bucket  gs://cpe-kmutt-leb1-rag-966535060840/  ...
Chunking of data in Public GCS Bucket done.


#### Import files direct context retrieval

In [11]:
# Direct context retrieval. Prints all results that pass threshold of 0.5
response = rag.retrieval_query(
    rag_resources=[
        rag.RagResource(
            rag_corpus=rag_corpus.name,
            # Optional: supply IDs from `rag.list_files()`.
            # rag_file_ids=["rag-file-1", "rag-file-2", ...],
        )
    ],
    rag_retrieval_config=rag.RagRetrievalConfig(
        top_k=20,  # Optional
        filter=rag.Filter(
            vector_distance_threshold=0.5,  # Optional
        ),
    ),
    #text="What is RAG and why it is helpful?",
    # text = "How many credits in masters degree program at CPE, KMUTT"
    text = "How many total credits are required for the B.Eng. in Computer Engineering (Regular Program)?"
)
print(response)

# Optional: The retrieved context can be passed to any SDK or model generation API to generate final results.
# context = " ".join([context.text for context in response.contexts.contexts]).replace("\n", "")

contexts {
  contexts {
    source_uri: "gs://cpe-kmutt-leb1-rag-966535060840/วศ.ม._วท.ม.วิศวกรรมคอมพิวเตอร์ (หลักสูตรนานาชาติ)-MEng_MSc International_2565.pdf"
    text: "276 (3 Aug 2022)\r\nNote: 1. Student cannot receive more than 5 thesis credits before passing the proposal \r\nexam.\r\n2. Students cannot receive more than 22 credits before a paper is submitted.\r\n3. Student can register in 1 or 2 electives credits but the total credit \r\nthroughout the program must be at least 37 credits.\r\n4. The proposal defense and the final defense cannot take place in the same \r\nsemester. \r\nPlan A type A2 (12-credit Thesis)\r\nYear 1 Semester 1\r\nCPE 600 Research Methodology and Technical Research \r\nWriting\r\n3 (3-0-9)\r\nCPE 601 Computer Engineering Seminar 1 (0-2-3)\r\nCPE 6xx Elective I 3 (3-0-9)\r\nCPE 6xx Elective II 3 (3-0-9)\r\nTotal 10 (9-2-30)\r\nHours per week 41\r\nAccumulated credits 10\r\nYear 1 Semester 2\r\nCPE 6xx Elective I 3 (3-0-9)\r\nCPE 6xx Elective II 3 (3-0-9

#### Create RAG Retrieval Tool

In [12]:
# Create a tool for the RAG Corpus
rag_retrieval_tool = Tool(
    retrieval=Retrieval(
        vertex_rag_store=VertexRagStore(
            rag_corpora=[rag_corpus.name],
            similarity_top_k=10,
            vector_distance_threshold=0.5,
        )
    )
)

#### Generate Content with Gemini using RAG Retrieval Tool

In [14]:
#MODEL_ID = "gemini-2.0-flash-001"
#MODEL_ID = "gemini-2.5-pro"
MODEL_ID = "gemini-2.5-flash"

In [15]:
response = client.models.generate_content( # Call the generate_content method from the client's models.
    model=MODEL_ID, # Specify the generative model to use (e.g., 'gemini-2.5-pro').
    #contents="What pack is most popular at AIS", # An example of a commented-out content query.
    contents="How many credits in undergraduate degree program at CPE, KMUTT", # The user's query passed to the model.
    config=GenerateContentConfig(tools=[rag_retrieval_tool]), # Configure content generation, including the RAG retrieval tool.
)

display(Markdown(response.text)) # Display the generated response as Markdown.


The provided sources describe the Master of Engineering (MEng) and Master of Science (MSc) International programs in Computer Engineering at KMUTT, not an undergraduate degree program. Therefore, the number of credits for an undergraduate degree program at CPE, KMUTT is not available in these sources.

### 3. Retrieval & Answer Generation

#### Chat with the RAG-enabled Model

- **EVALUATION PART I**\
Use the four current degree programs documents of CPE, KMUTT for B.Eng. Regular, B.Eng.
International, Masters, and Ph.D. programs. Put them in your GCP Cloud Storage as buckets (not on
Google Drive). Then one at a time (not in a chat stream) test at least 17 cases

In [16]:
from vertexai import generative_models
from vertexai import rag

# Re-create a tool for the RAG Corpus specifically for the chat model
# This ensures compatibility with vertexai.generative_models.GenerativeModel
rag_retrieval_tool_for_chat = generative_models.Tool.from_retrieval(
    retrieval=rag.Retrieval(
        source=rag.VertexRagStore(
            rag_resources=[rag.RagResource(rag_corpus=rag_corpus.name)],
            rag_retrieval_config=rag.RagRetrievalConfig(
                top_k=10,  # Optional
                filter=rag.Filter(
                    vector_distance_threshold=0.5,  # Optional
                ),
            ),
        )
    )
)

# First, instantiate the generative model with the RAG retrieval tool
model_for_chat = generative_models.GenerativeModel(MODEL_ID, tools=[rag_retrieval_tool_for_chat])

/usr/local/lib/python3.12/dist-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


In [17]:
# Then, start the chat session with no history at first
chat_session = model_for_chat.start_chat(history=[])

print(f"Chatting with {MODEL_ID}. Type 'end' to exit.")

input_list = [
    # ====================== The Fact-Check ======================
    "How many total credits are required to graduate from the B.Eng. in Computer Engineering (Regular Program)?",
    "What are the specific degree requirements for a candidate to enroll in the CPE Master's program?",
    "List the required Mathematics (MTH) courses for the undergraduate Regular program.",
    "Which LNG courses must a student take to fulfill the graduation requirements for the International B.Eng. program?",
    "In the Master's Program Plan 1.2, how many credits are allocated for the Dissertation?",
    "How many General Physics courses are required in the undergraduate curriculum?",
    "What is the maximum number of years allowed to complete the Ph.D. program?",
    "Is an internship or industrial training a mandatory requirement for the B.Eng. International program?",
    "How many credits is the 'Special Project Study' worth in the Master's Plan 2.1?",
    "What are the names of the elective tracks available in the 4th year of the B.Eng. Regular program?",
    # ====================== The Out-of-Bounds ======================
    "What is the capital of Thailand?",
    "What was Google's total profit for the last fiscal year?",
    "What are the ingredients needed to cook authentic Thai Tom Yum Goong?",
    "How many campuses does King Monkut's University of Technology Thonburi (KMUTT) have?"
    # ====================== The Ambiguity ======================
    "Compare the total credit requirements for graduation between the B.Eng. Regular program and the B.Eng. International program.",
    "What is the difference in the degree abbreviations (Thai and English) between the Ph.D. and the Master of Engineering programs?",
    "Which specific core course code (e.g., CPE XXX) is shared as a requirement between both the Ph.D. program and the Master's program?",
    # end
    "end"
]

for user_input in input_list:
    # user_input = input("You: ")
    if user_input.lower() == "end":
        print("Chat ended.")
        break
    display(Markdown(f"**Question:** {user_input}"))
    # Send the user's message and get the model's response
    response = chat_session.send_message(user_input)
    display(Markdown(f"**Model:** {response.text}"))

Chatting with gemini-2.5-flash. Type 'end' to exit.


**Question:** How many total credits are required to graduate from the B.Eng. in Computer Engineering (Regular Program)?

**Model:** The total credits required to graduate from the B.Eng. in Computer Engineering (Regular Program) is 130 credits.

**Question:** What are the specific degree requirements for a candidate to enroll in the CPE Master's program?

**Model:** For the CPE Master's program (MEng/MSc International), a candidate must achieve a TETET score of 5.0, or other equivalent standard English tests, to be approved for the final thesis or project defense. The provided sources describe various course offerings and thesis/professional project credit requirements for program completion, but do not specify academic degree requirements for initial enrollment.

...

**Question:** What is the difference in the degree abbreviations (Thai and English) between the Ph.D. and the Master of Engineering programs?

**Model:** The degree abbreviations are as follows:

**Ph.D. (Doctor of Philosophy Program in Electrical and Computer Engineering)**
*   **Thai Abbreviation:** ปร.ด. (วิศวกรรมไฟฟ้าและคอมพิวเตอร์)
*   **English Abbreviation:** Ph.D. (Electrical and Computer Engineering)

**Master of Engineering (Computer Engineering)**
*   **Thai Abbreviation:** วศ.ม. (วิศวกรรมคอมพิวเตอร์)
*   **English Abbreviation:** M.Eng. (Computer Engineering)

**Question:** Which specific core course code (e.g., CPE XXX) is shared as a requirement between both the Ph.D. program and the Master's program?

**Model:** Based on the provided information, there is no specific core course code that is shared as a requirement between both the Ph.D. program and the Master's program.

Chat ended.


- **EVALUATION PART II**\
Use only the Thai version of B.Eng. curriculum. List 5 queries as ground truth. List all the
chunks and find out the chunks that contain the answers to them as your ground truth (you can use
LLM to help you). For each of the following 4 combinations of {text-embedding-005,
text-multilingual-embedding-002} x {"gemini-2.5-pro", "gemini-2.5-flash"}, report the
Recall@k for k = 10, for each of the 5 queries and the Overall Recall@k. Note: Do not set a threshold.

In [18]:
rag_corpus_005 = rag.create_corpus(
    display_name="my-rag-corpus",
    backend_config=rag.RagVectorDbConfig(
        rag_embedding_model_config=rag.RagEmbeddingModelConfig(
            vertex_prediction_endpoint=rag.VertexPredictionEndpoint(
                publisher_model=EMBEDDING_MODEL_005
            )
        )
    ),
)

rag_corpus_002 =  rag.create_corpus(
    display_name="my-rag-corpus",
    backend_config=rag.RagVectorDbConfig(
        rag_embedding_model_config=rag.RagEmbeddingModelConfig(
            vertex_prediction_endpoint=rag.VertexPredictionEndpoint(
                publisher_model=EMBEDDING_MODEL_002
            )
        )
    ),
)

In [19]:
print("Chunking of file test.md ...")
rag_file_005 = rag.upload_file(
    corpus_name=rag_corpus_005.name,
    path="test.md",
    display_name="test.md",
    description="my test file",
    #next line on transformation_config is optional, so it's been removed to use the default.
    #transformation_config=rag.TransformationConfig(
    #    chunking_config=rag.ChunkingConfig(chunk_size=512, chunk_overlap=50))
)

rag_file_002 = rag.upload_file(
    corpus_name=rag_corpus_002.name,
    path="test.md",
    display_name="test.md",
    description="my test file",
    #next line on transformation_config is optional, so it's been removed to use the default.
    #transformation_config=rag.TransformationConfig(
    #    chunking_config=rag.ChunkingConfig(chunk_size=512, chunk_overlap=50))
)
print("Chunking of test.md done.")

Chunking of file test.md ...
Chunking of test.md done.


In [20]:
# 1. Create the folder first so gsutil knows it's a directory
!mkdir -p regular_pdfs

# 2. Copy the files from the public bucket into that folder
# We use quotes and the star (*) to ensure we get the files inside the folder
!gsutil -m cp -r "gs://cpe-kmutt-leb1-rag-966535060840/วศ.บ.วิศวกรรมคอมพิวเตอร์-BE Regular TH_2564.pdf" ./regular_pdfs/

# 3. Verify the files are there
!ls ./regular_pdfs


Copying gs://cpe-kmutt-leb1-rag-966535060840/วศ.บ.วิศวกรรมคอมพิวเตอร์-BE Regular TH_2564.pdf...
- [1/1 files][404.9 KiB/404.9 KiB] 100% Done                                    
Operation completed over 1 objects/404.9 KiB.                                    
'วศ.บ.วิศวกรรมคอมพิวเตอร์-BE Regular TH_2564.pdf'


In [21]:
#reading from a public bucket.  This bucket is hosted by someone else so it's free.
INPUT_GCS_BUCKET = "gs://cpe-kmutt-leb1-rag-966535060840/วศ.บ.วิศวกรรมคอมพิวเตอร์-BE Regular TH_2564.pdf"

print("Start reading and chunking data in gcs bucket ", INPUT_GCS_BUCKET, " ...")

response_005 = rag.import_files(
    corpus_name=rag_corpus_005.name,
    paths=[INPUT_GCS_BUCKET],
    # Optional
    transformation_config=rag.TransformationConfig(
        chunking_config=rag.ChunkingConfig(chunk_size=1024, chunk_overlap=128)
    ),
    # max_embedding_requests_per_min=900,  # Optional
)

response_002 = rag.import_files(
    corpus_name=rag_corpus_002.name,
    paths=[INPUT_GCS_BUCKET],
    # Optional
    transformation_config=rag.TransformationConfig(
        chunking_config=rag.ChunkingConfig(chunk_size=1024, chunk_overlap=128)
    ),
    # max_embedding_requests_per_min=900,  # Optional
)

print("Chunking of data in Public GCS Bucket done.")

Start reading and chunking data in gcs bucket  gs://cpe-kmutt-leb1-rag-966535060840/วศ.บ.วิศวกรรมคอมพิวเตอร์-BE Regular TH_2564.pdf  ...
Chunking of data in Public GCS Bucket done.


In [25]:
import json
import re
import pandas as pd
from vertexai.generative_models import GenerativeModel

# Auditor model for Recall checking — fixed: was incorrectly set to gemini-1.5-flash
model_2_5_flash = GenerativeModel("gemini-2.5-flash")

def get_ai_recall_score(query, retrieved_text, ground_truth_answer):
    """Checks if the answer is present in the text without knowing chunk IDs."""
    prompt = f"""
    You are an academic auditor. 
    Question: {query}
    Correct Answer: {ground_truth_answer}
    
    Retrieved Text: 
    {retrieved_text}
    
    Task: Is the Correct Answer found within the Retrieved Text?
    Respond ONLY with a JSON: {{"score": 1}} if yes, {{"score": 0}} if no.
    """
    try:
        response = model_2_5_flash.generate_content(prompt).text
        cleaned = re.sub(r'```json|```', '', response).strip()
        return json.loads(cleaned).get("score", 0)
    except:
        return 0

def run_lab_evaluation_v4(queries_list):
    embedding_configs = [
        {"label": "text-embedding-005", "corpus": rag_corpus_005.name},
        {"label": "text-multilingual-embedding-002", "corpus": rag_corpus_002.name}
    ]
    
    recall_summary = {}  # { embedding_label: [recall_per_query] }

    for config in embedding_configs:
        print(f"\n\nEMBEDDING: {config['label']}")
        recall_summary[config['label']] = []
        
        for i, item in enumerate(queries_list, 1):
            print(f"\nQuery {i}: {item['query']}")
            
            # 1. Retrieval (Top-K = 10, no threshold so all top-10 are returned)
            response = rag.retrieval_query(
                rag_resources=[rag.RagResource(rag_corpus=config['corpus'])],
                rag_retrieval_config=rag.RagRetrievalConfig(top_k=10),
                text=item['query']
            )
            
            table_data = []
            hit = 0  # Recall@10: 1 if answer found in ANY of top-10 chunks

            for rank, ctx in enumerate(response.contexts.contexts, 1):
                chunk_text = ctx.text.strip()

                # Fix: use source_uri + rank as chunk identifier (source_id doesn't exist)
                source_uri = getattr(ctx, 'source_uri', '')
                source_name = source_uri.split('/')[-1] if source_uri else 'unknown'
                chunk_id = f"chunk-{rank} [{source_name}]"

                # Fix: per-chunk AI check for accurate Is-Find flag
                per_chunk_score = get_ai_recall_score(item['query'], chunk_text, item['gt_answer'])
                found_flag = "Yes" if per_chunk_score == 1 else "No"
                if per_chunk_score == 1:
                    hit = 1

                # Display 2-3 lines (~150 chars)
                table_data.append({
                    "Rank": rank,
                    "Is Find": found_flag,
                    "Chunk ID": chunk_id,
                    "Retrieved Text": chunk_text[:150].replace('\n', ' ') + "..."
                })

            # 2. Print the formatted table for this query
            df = pd.DataFrame(table_data)
            print(df.to_markdown(index=False))

            # 3. Recall@10 for this query
            recall_summary[config['label']].append(hit)
            print(f"\nRecall@10 for Query {i}: {hit}")
            print("-" * 80)

    # 4. Print overall Recall@10 summary table
    print("\n\n===== Recall@10 Summary =====")
    summary_rows = []
    for label, scores in recall_summary.items():
        row = {"Embedding": label}
        for j, s in enumerate(scores, 1):
            row[f"Q{j} Recall@10"] = s
        row["Overall Recall@10"] = f"{sum(scores)/len(scores):.2f}"
        summary_rows.append(row)
    summary_df = pd.DataFrame(summary_rows)
    print(summary_df.to_markdown(index=False))


# Ground truth data from the Thai curriculum
queries = [
    {"query": "จำนวนหน่วยกิตรวมตลอดหลักสูตรคือเท่าไร", "gt_answer": "130 หน่วยกิต"},
    {"query": "วิชาบังคับก่อนของ CPE 112 คืออะไร", "gt_answer": "CPE 100"},
    {"query": "คะแนน O-Net เท่าไรถึงยกเว้น LNG 120", "gt_answer": "สูงกว่า 40 คะแนน"},
    {"query": "วิชาบังคับก่อนของ STA 302 คืออะไร", "gt_answer": "MTH 102 คณิตศาสตร์ 2"},
    {"query": "หน่วยกิตของ CPE 401 คืออะไร", "gt_answer": "3 (0-6-9) หน่วยกิต"}
]

run_lab_evaluation_v5(queries)



EMBEDDING: text-embedding-005

Query 1: จำนวนหน่วยกิตรวมตลอดหลักสูตรคือเท่าไร
|   Rank | Is Find   | Chunk ID                                         | Retrieve Text                                                                                  |
|-------:|:----------|:-------------------------------------------------|:-----------------------------------------------------------------------------------------------|
|      1 | No        | วศ.บ.วิศวกรรมคอมพิวเตอร์-BE Regular TH_2564.pdf_R1  | ผลลัพธ์การเรียนรู้ (Learning Outcomes)                                                              |
|        |           |                                                  |  1. Identify purposes, main ideas and important details of texts on academic topics.           |
|        |           |                                                  |  2. Interact with others i...                                                                  |
|      2 | No        | วศ.บ.วิศวกรรมคอมพิวเตอร์-BE Regula

#### DELETE CORPUS TO AVOID CHARGES

In [26]:
# Print the name to be sure
print(f"Deleting Corpus: {rag_corpus.name}, {rag_corpus_002.name}, {rag_corpus_005}")
# Delete the corpus to stop the hourly billing
rag.delete_corpus(name=rag_corpus.name)
rag.delete_corpus(name=rag_corpus_002.name)
rag.delete_corpus(name=rag_corpus_005.name)

Deleting Corpus: projects/966535060840/locations/asia-southeast1/ragCorpora/3379951520341557248, projects/966535060840/locations/asia-southeast1/ragCorpora/137359788634800128, RagCorpus(name='projects/966535060840/locations/asia-southeast1/ragCorpora/6838716034162098176', display_name='my-rag-corpus', description='', vertex_ai_search_config=None, backend_config=RagVectorDbConfig(vector_db=RagManagedDb(), rag_embedding_model_config=RagEmbeddingModelConfig(vertex_prediction_endpoint=VertexPredictionEndpoint(endpoint=None, publisher_model='projects/local-disk-478816-f6/locations/asia-southeast1/publishers/google/models/text-embedding-005', model=None, model_version_id=None))), encryption_spec=)
Successfully deleted the RagCorpus.
Successfully deleted the RagCorpus.
Successfully deleted the RagCorpus.
